In [ ]:
def dNdlogD(N,x,mu,sigma):
    return N*np.exp(-(np.log10(x) - np.log10(mu*2))**2 / (2 * np.log10(sigma)**2))/ (np.log10(sigma) * np.sqrt(2 * np.pi))  


def dNdlogD_dask(N, x, mu, sigma):
    """
    Compute dNdlogD using dask.array for lazy evaluation.
    """
    log_term = (da.log10(x) - da.log10(mu * 2)) ** 2
    denom = 2 * (da.log10(sigma) ** 2)
    exponent = -log_term / denom
    return N * da.exp(exponent) / (da.log10(sigma) * np.sqrt(2 * np.pi))

In [ ]:
PNSD_ds = xr.Dataset()
Xspace = xr.DataArray(np.logspace(-0.5,3, num=50), dims =['R'], coords= {'R':np.logspace(-0.5,3, num=50)})
for i in range(0, 16):
    sig, nmr, nconc = f"SIGMA{i:02d}", f"NMR{i:02d}", f"NCONC{i:02d}"
    if sig not in ds:
        continue
    PNSD_ds[sig] = ds[sig]
    PNSD_ds[nmr] = ds[nmr]
    PNSD_ds[nconc] = ds[nconc]

    PNSD_ds[f"dNdlogD{i:02d}"] = xr.apply_ufunc(
        dNdlogD,
        ds[nconc],
        Xspace,
        ds[nmr],
        ds[sig],
        dask="parallelized",
        dask_gufunc_kwargs={'allow_rechunk': True},
        output_dtypes=[float],
    )

In [ ]:
dNdlogD_vars = [v for v in PNSD_ds.data_vars if v.startswith("dNdlogD")]
dNdlogD = sum(PNSD_ds[v] for v in dNdlogD_vars)


In [ ]:
dNdlogD = dNdlogD.compute()

In [ ]:
fig, ax1 = plt.subplots()

# ---- Left y-axis: susceptibility & correlation ----
ax1.plot(
    radii * 2,
    reg_ds['slope'].sel(station='SMR-II'),
    label='Susceptibility'
)

ax1.fill_between(
    radii * 2,
    reg_ds['slope'].sel(station='SMR-II')
    - 1.98 * reg_ds['std_err'].sel(station='SMR-II'),
    reg_ds['slope'].sel(station='SMR-II')
    + 1.98 * reg_ds['std_err'].sel(station='SMR-II'),
    alpha=0.5,
    label='95% Confidence Interval'
)

ax1.plot(
    radii * 2,
    reg_ds['r_value'].sel(station='SMR-II'),
    '--',
    label='Correlation (r)'
)

ax1.set_xlabel('Particle Diameter Dₚ (nm)')
ax1.set_ylabel('Susceptibility / Correlation')
ax1.set_ylim([-0.2, 1])
ax1.set_xscale('log')
ax1.set_xlim([1, 1000])

# ---- Right y-axis: dNdlogD ----
ax2 = ax1.twinx()

ax2.plot(
    dNdlogD['R'] * 2,
    dNdlogD.sel(station='SMR-II').isel(lev = slice(-1,-7,-1)).mean('lev').mean('time'),
    color='black',
    label='dN/dlogD'
)
ax2.fill_between(
    dNdlogD['R'] * 2,
    dNdlogD.sel(station='SMR-II').isel(lev = slice(-1,-7,-1)).mean('lev').quantile(.25, dim = 'time'),
    dNdlogD.sel(station='SMR-II').isel(lev = slice(-1,-7,-1)).mean('lev').quantile(.75, dim = 'time'),
    label = 'dN/dlogD IQR',
    color = 'black',
    alpha = 0.2
)

ax2.set_ylabel('dN / dlogD')
ax2.set_yscale('log')
ax2.set_ylim([.1, 5000])


# ---- Title ----
plt.title(
    f'CCN–CDNC Susceptibility (SMR-II)'
)

# ---- Combined legend ----
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(
    lines1 + lines2,
    labels1 + labels2,
    bbox_to_anchor=(0.5, 0.4)
)
plt.show()

In [ ]:
stations = reg_ds.station.values
n_stations = len(stations)

fig, axes = plt.subplots(
    4, 4,
    figsize=(14, 14),
    sharex=True,
    sharey=True
)

axes = axes.flatten()

for i, station in enumerate(stations):

    ax = axes[i]

    slope = reg_ds['slope'].sel(station=station)
    std_err = reg_ds['std_err'].sel(station=station)
    r_val = reg_ds['r_value'].sel(station=station)

    # ---- Left axis: susceptibility & correlation ----
    ax.plot(radii * 2, slope, label='Susceptibility')
    ax.fill_between(
        radii * 2,
        slope - 1.96 * std_err,
        slope + 1.96 * std_err,
        alpha=0.5,
        label='95% Confidence Interval'
    )
    ax.plot(
        radii * 2,
        r_val,
        '--',
        label='Correlation (r)'
    )

    ax.set_title(station, fontsize=10)
    ax.set_xscale('log')
    ax.set_xlim([1, 1000])
    ax.set_ylim([-0.2, 1])

    # ---- Right axis: PNSD ----
    ax2 = ax.twinx()
    ax2.plot(
        dNdlogD['R'] * 2,
        dNdlogD.sel(station=station).isel(lev = slice(-1,-7,-1)).mean('lev').mean('time'),
        color='black',
        alpha=0.6,
        label='PNSD'
    )
    ax2.fill_between(
    dNdlogD['R'] * 2,
    dNdlogD.sel(station=station).isel(lev = slice(-1,-7,-1)).mean('lev').quantile(.25, dim = 'time'),
    dNdlogD.sel(station=station).isel(lev = slice(-1,-7,-1)).mean('lev').quantile(.75, dim = 'time'),
    label = 'dN/dlogD IQR',
    color = 'black',
    alpha = 0.2
)

    # Keep right axis ticks subtle
    ax2.tick_params(axis='y', labelsize=8)
    ax2.set_ylabel('')  # no per-panel label
    ax2.set_yscale('log')
    ax2.set_ylim([10, 5000])

# --- Last panel used only for legend ---
legend_ax = axes[-1]
legend_ax.axis('off')

# Collect handles from one subplot (includes both axes)
handles1, labels1 = axes[0].get_legend_handles_labels()
handles2, labels2 = axes[0].twinx().get_legend_handles_labels()

legend_ax.legend(
    handles1 + handles2,
    labels1 + labels2,
    loc='center',
    frameon=False,
    fontsize=10
)

# Axis labels (only on outer edges)
for ax in axes[12:16]:
    ax.set_xlabel('Particle Diameter $D_p$ (nm)')

for ax in axes[::4]:
    ax.set_ylabel('Susceptibility / Correlation')

# Optional single right-axis label
fig.text(0.92, 0.5, 'dN / dlogD', va='center', rotation=-90)

plt.tight_layout(rect=[0, 0, 0.9, 1])
plt.show()

In [ ]:
mesh = plt.pcolormesh(
    PNSD_ds['R'] * 2,
    height.sel(station='SGP'),
    dNdlogD.sel(station='SGP').mean('time'),
    shading='nearest',
    cmap='viridis',
    vmin = 0,
    vmax = 5000
)

plt.xscale('log')
plt.xlim([3, 400])
plt.ylim([200, 2500])
plt.xticks([3, 10, 50, 100, 200], ['3', '10', '50', '100', '200'])

plt.colorbar(mesh, label='dN/dlogD')
plt.xlabel('Diameter (nm)')
plt.ylabel('Height (m)')
plt.title('Aerosol Size Distribution Year round (SGP)')
plt.tight_layout()

In [ ]:
spring = dNdlogD.sel(
    station='SGP',
    time=dNdlogD['time'].dt.month.isin([3, 4, 5])
).mean('time')

summer = dNdlogD.sel(
    station='SGP',
    time=dNdlogD['time'].dt.month.isin([6, 7, 8])
).mean('time')


In [ ]:
plt.figure(figsize=(7,5))

mesh = plt.pcolormesh(
    PNSD_ds['R'] * 2,
    height.sel(station='SGP'),
    spring,
    shading='nearest',
    cmap='viridis',
    vmin=0,
    vmax=5000
)

plt.xscale('log')
plt.xlim([3, 400])
plt.ylim([200, 2500])
plt.xticks([3, 10, 50, 100, 200], ['3', '10', '50', '100', '200'])

plt.colorbar(mesh, label='dN/dlogD ($cm ^{-3}$)')
plt.xlabel('Diameter (nm)')
plt.ylabel('Height (m)')
plt.title('Spring Aerosol Size Distribution (SGP)')
plt.tight_layout()

In [ ]:
plt.figure(figsize=(7,5))

mesh = plt.pcolormesh(
    PNSD_ds['R'] * 2,
    height.sel(station='SGP'),
    summer,
    shading='nearest',
    cmap='viridis',
    vmin=0,
    vmax=5000
)

plt.xscale('log')
plt.xlim([3, 400])
plt.ylim([200, 2500])
plt.colorbar(mesh, label='dN/dlogD')
plt.xlabel('Diameter (nm)')
plt.xticks([3, 10, 50, 100, 200], ['3', '10', '50', '100', '200'])

plt.ylabel('Height (m)')
plt.title('Summer Aerosol Size Distribution (SGP)')
plt.tight_layout()
